In [1]:
!hostname

c-11-03


In [2]:
import sys
import gzip
import glob
import numpy as np
import pandas as pd

import matplotlib
from matplotlib import pyplot as plt

In [3]:
def load_mgatk_output(output_dir, mito_length):
    # assuming mgatk output naming convention
    base_files = [glob.glob(output_dir + "*.{}.txt.gz".format(nt))[0] for nt in "ATCG"]

    base_coverage_dict = dict()
    for i, nt in enumerate("ATCG"):
        cur_base_data = pd.read_csv(gzip.open(base_files[i]), header=None)

        # gather coverage for forward strand
        fwd_base_df = cur_base_data[[0, 1, 2]].pivot_table(index=1, columns=0)
        fwd_base_df.columns = [x[1] for x in fwd_base_df.columns.values]  # flatten weird multiindex after pivot
        fwd_base_df.index.name = None
        # missing_pos = [x for x in range(1, mito_length + 1) if x not in fwd_base_df.columns]
        # fwd_base_df[missing_pos] = 0  # fill in missing positions
        all_columns = [x for x in range(1, mito_length + 1)]
        fwd_base_df = fwd_base_df.reindex(columns=all_columns, fill_value=0)
        fwd_base_df = fwd_base_df.fillna(0).sort_index(axis=1)  # assume all nan are true zeroes

        # gather coverage for forward strand
        rev_base_df = cur_base_data[[0, 1, 3]].pivot_table(index=1, columns=0)
        rev_base_df.columns = [x[1] for x in rev_base_df.columns.values]
        rev_base_df.index.name = None
        # missing_pos = [x for x in range(1, mito_length + 1) if x not in rev_base_df.columns]
        # rev_base_df[missing_pos] = 0
        all_columns = [x for x in range(1, mito_length + 1)]
        rev_base_df = rev_base_df.reindex(columns=all_columns, fill_value=0)
        rev_base_df = rev_base_df.fillna(0).sort_index(axis=1)

        # organize base data into a dict
        base_coverage_dict[nt] = (fwd_base_df, rev_base_df)

    return base_coverage_dict


def gather_possible_variants(base_coverage_dict, reference_file):
    # sum across cells and strands for each base and position
    aggregated_genotype = pd.DataFrame(np.zeros((4, mito_length)), index=list("ATCG"), columns=np.arange(1, mito_length + 1))
    for nt in base_coverage_dict:
        # sum across cells for each strand separately
        fwd_base_df, rev_base_df = base_coverage_dict[nt]
        fwd_base_sum, rev_base_sum = fwd_base_df.sum(), rev_base_df.sum()

        # sequencing artifact if a base/position is only nonzero for one strand across cells, ignore them
        masking = ~((fwd_base_sum > 0) & (rev_base_sum > 0))  # True if position not >0 for both strands
        fwd_base_sum[masking], rev_base_sum[masking] = 0, 0

        # sum across strands
        aggregated_genotype.loc[nt, :] = fwd_base_sum + rev_base_sum

    # make a reference set of tuples (pos, ref_base)
    ref_set = [x.strip().split() for x in open(reference_file, "r").readlines()]
    ref_N_positions = [int(x[0]) for x in ref_set if x[1].upper() not in letters]
    ref_set = set([(int(x[0]), x[1].upper()) for x in ref_set if x[1].upper() in letters])
    ref_dict = dict(ref_set)

    # make an observed set of tuples which are nonzero
    non_zero_idx = np.where(aggregated_genotype > 0)
    non_zero_bases = [letters[i] for i in non_zero_idx[0]]
    non_zero_pos = [int(i + 1) for i in non_zero_idx[1]]
    observed_set = list(zip(non_zero_pos, non_zero_bases))
    observed_set = set([x for x in observed_set if x[0] not in ref_N_positions])  # disregard positions in ref with N

    # take difference between observed and reference
    variant_set = observed_set - ref_set
    variants = sorted([(x[0], ref_dict[x[0]], x[1]) for x in list(variant_set)], key=lambda x: x[0])  # (pos, ref base, obs base)

    return variants

In [4]:
# MGATK_OUT_DIR = sys.argv[1]
MGATK_OUT_DIR = "/home/liuc9/github/scMOCHA/05-Liming/scmocha-mixed-cellline-high-depth2/cromwell-executions/scMOCHA/139358d8-df39-4274-b931-9c42b8d9c3bb/call-call_mt_variants/execution/cluster/final/"
# sample_prefix = sys.argv[2]
sample_prefix = "cluster"
# mito_length = int(sys.argv[3])  # 16569
mito_length = 16569
# low_coverage_threshold = int(sys.argv[4])  # 10
low_coverage_threshold = 10
# mito_genome = sys.argv[5]  # chrM
mito_genome = "MT"

In [15]:
letters = list("ATCG")

base_coverage_dict = load_mgatk_output(MGATK_OUT_DIR, mito_length)
cell_barcodes = base_coverage_dict["A"][0].index

In [22]:
# total coverage per position per cell
total_coverage = pd.DataFrame(np.zeros((len(cell_barcodes), mito_length)), index=cell_barcodes, columns=np.arange(1, mito_length + 1))


In [23]:
total_coverage

,1,2,3,4,5,6,7,8,9,10,...,16560,16561,16562,16563,16564,16565,16566,16567,16568,16569
cluster_0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cluster_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cluster_2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cluster_3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cluster_4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
for nt in base_coverage_dict:
    total_coverage += base_coverage_dict[nt][0]
    total_coverage += base_coverage_dict[nt][1]

In [35]:
total_coverage.iloc[:, 6775]

cluster_0    20.0
cluster_1     0.0
cluster_2     9.0
cluster_3     8.0
cluster_4     8.0
Name: 6776, dtype: float64

In [36]:
# exclude low coverage cells from variant calling
cell_barcodes = total_coverage.index[total_coverage.mean(axis=1) > low_coverage_threshold]


In [39]:

for nt in base_coverage_dict:
    base_coverage_dict[nt] = (base_coverage_dict[nt][0].loc[cell_barcodes, :], base_coverage_dict[nt][1].loc[cell_barcodes, :])
total_coverage = total_coverage.loc[cell_barcodes, :]

In [40]:
base_coverage_dict

{'A': (           1      2      3      4      5      6      7      8      9      \
  cluster_0      0   28.0      0      0   31.0      0   37.0      0    0.0   
  cluster_1      0    8.0      0      0    9.0      0   11.0      0    0.0   
  cluster_2      0   15.0      0      0   17.0      0   20.0      0    0.0   
  cluster_3      0   19.0      0      0   24.0      0   26.0      0    0.0   
  cluster_4      0   15.0      0      0   18.0      0   20.0      0    0.0   
  
             10     ...  16560  16561  16562  16563  16564  16565  16566  16567  \
  cluster_0    0.0  ...      0   40.0      0      0   37.0      0      0   33.0   
  cluster_1    0.0  ...      0   22.0      0      0   20.0      0      0   18.0   
  cluster_2    0.0  ...      0   41.0      0      0   38.0      0      0   35.0   
  cluster_3    0.0  ...      0   39.0      0      0   36.0      0      0   33.0   
  cluster_4    0.0  ...      0   38.0      0      0   35.0      0      0   32.0   
  
             16568  165

In [41]:
# call potential variants
variants = gather_possible_variants(base_coverage_dict, MGATK_OUT_DIR + mito_genome + "_refAllele.txt")

variant_names = ["{}{}>{}".format(x[0], x[1], x[2]) for x in variants]

In [43]:
variant_names

['58T>C',
 '64C>T',
 '72T>A',
 '73A>G',
 '146T>C',
 '152T>C',
 '195T>C',
 '198C>T',
 '225G>T',
 '226T>C',
 '263A>G',
 '282T>A',
 '292T>A',
 '293T>C',
 '295C>T',
 '295C>G',
 '299C>A',
 '302A>C',
 '303C>T',
 '303C>A',
 '304C>A',
 '305C>T',
 '308C>T',
 '309C>T',
 '310T>C',
 '311C>T',
 '312C>T',
 '315C>T',
 '316G>C',
 '342T>A',
 '356C>A',
 '357A>C',
 '380G>A',
 '410G>C',
 '447C>G',
 '592C>T',
 '658G>T',
 '731A>G',
 '750A>G',
 '769G>A',
 '775C>T',
 '887G>A',
 '916C>T',
 '929A>G',
 '933G>A',
 '961T>A',
 '1018G>A',
 '1042T>C',
 '1438A>G',
 '1465C>A',
 '1598G>C',
 '1602C>A',
 '1604G>A',
 '1768G>T',
 '1770G>A',
 '1784A>G',
 '1883G>A',
 '1894G>T',
 '1949G>A',
 '1987G>A',
 '2071T>A',
 '2149G>A',
 '2157T>A',
 '2212C>T',
 '2240C>A',
 '2262C>T',
 '2330T>A',
 '2408T>C',
 '2416T>C',
 '2469A>G',
 '2535A>G',
 '2617A>C',
 '2617A>G',
 '2617A>T',
 '2706A>G',
 '2789C>T',
 '2809C>T',
 '2838A>G',
 '2846G>A',
 '2871T>C',
 '3000A>G',
 '3010G>A',
 '3072T>C',
 '3076A>G',
 '3097T>C',
 '3109T>C',
 '3110C>A',
 '3197

In [50]:
variants[:5], variant_names[:5]

([(58, 'T', 'C'),
  (64, 'C', 'T'),
  (72, 'T', 'A'),
  (73, 'A', 'G'),
  (146, 'T', 'C')],
 ['58T>C', '64C>T', '72T>A', '73A>G', '146T>C'])

In [51]:
total_coverage[6776]

cluster_0    20.0
cluster_1     0.0
cluster_2     9.0
cluster_3     8.0
cluster_4     8.0
Name: 6776, dtype: float64

In [44]:
# build two <cell by variant tables>, one for each strand
total_coverage_variant_df = []
fwd_cell_variant_df, rev_cell_variant_df = [], []
for i, var in enumerate(variants):
    var_name = variant_names[i]
    pos, base = var[0], var[2]
    total_coverage_variant_df.append(total_coverage[pos])
    fwd_cell_variant_df.append(base_coverage_dict[base][0][pos].values)
    rev_cell_variant_df.append(base_coverage_dict[base][1][pos].values)
total_coverage_variant_df = pd.DataFrame(np.array(total_coverage_variant_df).T, index=cell_barcodes, columns=variant_names)
fwd_cell_variant_df = pd.DataFrame(np.array(fwd_cell_variant_df).T, index=cell_barcodes, columns=variant_names)
rev_cell_variant_df = pd.DataFrame(np.array(rev_cell_variant_df).T, index=cell_barcodes, columns=variant_names)
all_cell_variant_df = fwd_cell_variant_df + rev_cell_variant_df

In [52]:
total_coverage_variant_df["6776T>C"]

cluster_0    20.0
cluster_1     0.0
cluster_2     9.0
cluster_3     8.0
cluster_4     8.0
Name: 6776T>C, dtype: float64

In [53]:
fwd_cell_variant_df["6776T>C"]

cluster_0    17.0
cluster_1     0.0
cluster_2     0.0
cluster_3     0.0
cluster_4     0.0
Name: 6776T>C, dtype: float64

In [54]:
rev_cell_variant_df["6776T>C"]

cluster_0    1.0
cluster_1    0.0
cluster_2    0.0
cluster_3    0.0
cluster_4    0.0
Name: 6776T>C, dtype: float64

In [55]:
all_cell_variant_df["6776T>C"]

cluster_0    18.0
cluster_1     0.0
cluster_2     0.0
cluster_3     0.0
cluster_4     0.0
Name: 6776T>C, dtype: float64

In [56]:
# heteroplasmic ratio
heteroplasmic_df = all_cell_variant_df / total_coverage_variant_df

In [57]:
heteroplasmic_df["6776T>C"]

cluster_0    0.9
cluster_1    NaN
cluster_2    0.0
cluster_3    0.0
cluster_4    0.0
Name: 6776T>C, dtype: float64

In [62]:
# strand correlation
mask_idx = (fwd_cell_variant_df + rev_cell_variant_df) == 0  # set 0 on both strands to nan to exclude from correlation calculation
fwd_cell_variant_df[mask_idx] = np.nan
rev_cell_variant_df[mask_idx] = np.nan
variant_strand_corr = fwd_cell_variant_df.corrwith(rev_cell_variant_df).round(3)

In [63]:
variant_strand_corr

58T>C      -1.000
64C>T      -1.000
72T>A         NaN
73A>G       0.980
146T>C      0.997
            ...  
16309A>G    1.000
16366C>T    0.999
16390G>A    0.999
16399A>G    1.000
16519T>C    0.985
Length: 459, dtype: float64

In [61]:
variant_strand_corr["6776T>C"]

nan

In [67]:
all_cell_variant_df.sum(), heteroplasmic_df.var()

(58T>C         2.0
 64C>T         2.0
 72T>A         2.0
 73A>G       490.0
 146T>C      169.0
             ...  
 16309A>G    175.0
 16366C>T    185.0
 16390G>A    174.0
 16399A>G    155.0
 16519T>C    376.0
 Length: 459, dtype: float64,
 58T>C       0.000016
 64C>T       0.000014
 72T>A       0.000097
 73A>G       0.216640
 146T>C      0.188162
               ...   
 16309A>G    0.191046
 16366C>T    0.182429
 16390G>A    0.193644
 16399A>G    0.186394
 16519T>C    0.244351
 Length: 459, dtype: float64)

In [64]:
# vmr
variant_mean = all_cell_variant_df.sum() / total_coverage_variant_df.sum()
variant_var = heteroplasmic_df.var()
variant_vmr = variant_var / (variant_mean + 0.00000000001)

In [70]:
(heteroplasmic_df >= 0.05).sum()

58T>C       0
64C>T       0
72T>A       0
73A>G       4
146T>C      2
           ..
16309A>G    2
16366C>T    2
16390G>A    1
16399A>G    1
16519T>C    3
Length: 459, dtype: int64

In [71]:
# compute other summary stats
variant_positon = [x[0] for x in variants]
variant_nucleotide = ["{}>{}".format(x[1], x[2]) for x in variants]
variant_n_cells_conf_detected = ((fwd_cell_variant_df >= 2) & (rev_cell_variant_df >= 2)).sum()
variant_n_cells_over_5 = (heteroplasmic_df >= 0.05).sum()
variant_n_cells_over_10 = (heteroplasmic_df >= 0.1).sum()
variant_n_cells_over_20 = (heteroplasmic_df >= 0.2).sum()
variant_n_cells_over_95 = (heteroplasmic_df >= 0.95).sum()
max_heteroplasmy = heteroplasmic_df.max()
variant_mean_coverage = total_coverage_variant_df.mean()

In [72]:
# pack summary stats
variant_output = pd.DataFrame(
    [
        variant_positon,
        variant_nucleotide,
        variant_names,
        variant_vmr,
        variant_mean,
        variant_var,
        variant_n_cells_conf_detected,
        variant_n_cells_over_5,
        variant_n_cells_over_10,
        variant_n_cells_over_20,
        variant_n_cells_over_95,
        max_heteroplasmy,
        variant_strand_corr,
        variant_mean_coverage,
    ]
).T
variant_output.columns = ["position", "nucleotide", "variant", "vmr", "mean", "variance", "n_cells_conf_detected", "n_cells_over_5", "n_cells_over_10", "n_cells_over_20", "n_cells_over_95", "max_heteroplasmy", "strand_correlation", "mean_coverage"]
variant_output[["vmr", "mean", "variance", "strand_correlation", "mean_coverage", "max_heteroplasmy"]] = variant_output[["vmr", "mean", "variance", "strand_correlation", "mean_coverage", "max_heteroplasmy"]].astype(float)

In [74]:
variant_output.query("variant == '6776T>C'")

,position,nucleotide,variant,vmr,mean,variance,n_cells_conf_detected,n_cells_over_5,n_cells_over_10,n_cells_over_20,n_cells_over_95,max_heteroplasmy,strand_correlation,mean_coverage
171,6776,T>C,6776T>C,0.50625,0.4,0.2025,0,1,1,1,0,0.9,NaN,9.0
